In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
import joblib

# 1. Load the dataset
df = pd.read_csv("/content/hprime_clean.csv")

# 2. Define the Target (y)
y = df['meets_standard']

# 3. Drop Data Leakage and Unnecessary Text
columns_to_drop = [
    'candidate_id', 'assessment_date', 'assessment_number', 'assessor_name',
    'assessor_title', 'competency_areas', 'focus_area', 'case_summary',
    'feedback', 'meets_standard',
    'score_record_keeping', 'score_history_examination', 'score_management_plan',
    'score_clinical_judgement', 'score_history', 'score_physical_exam',
    'score_communication', 'score_clinical_judgement.1', 'score_professionalism',
    'score_organisation', 'score_overall_care', 'avg_score', 'score_std', 'any_score_1'
]
X = df.drop(columns=columns_to_drop)

# 4. Feature Selection: Separate Numeric and Categorical columns
# This matches the Week 6 Lab methodology
numeric_cols = X.select_dtypes(include=['float64', 'int64']).columns.to_list()
categorical_cols = X.select_dtypes(include=['object']).columns.to_list()

# 5. Handle Missing Values
# Fill missing 'days_since_last' with 0
X['days_since_last'] = X['days_since_last'].fillna(0)
# Fill numeric missing values with median
X[numeric_cols] = X[numeric_cols].fillna(X[numeric_cols].median())
# Fill categorical missing values with the mode
for col in categorical_cols:
    X[col] = X[col].fillna(X[col].mode()[0])

# 6. Split the Data BEFORE Transformation (Industry Best Practice)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 7. Apply UOWD Lab Standards: ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore', drop='if_binary'), categorical_cols)
    ])

# 8. Fit and Transform the Data
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

# 9. Save the preprocessor and data for Person 2
joblib.dump(preprocessor, 'hprime_preprocessor.pkl')

print("Part 1 Complete: Data successfully cleaned and transformed using ColumnTransformer!")
print(f"Training Data Shape: {X_train_processed.shape}")
print(f"Testing Data Shape: {X_test_processed.shape}")

Part 1 Complete: Data successfully cleaned and transformed using ColumnTransformer!
Training Data Shape: (1156, 19)
Testing Data Shape: (289, 19)


In [2]:

# Install SMOTE (Synthetic Minority Over-sampling Technique)
# Fixes the class imbalance: 1,102 passes vs only 54 fails in training data
import subprocess
subprocess.run(["pip", "install", "imbalanced-learn", "-q"])

# Imports
import numpy as np
import pandas as pd
import joblib
import warnings
warnings.filterwarnings("ignore")

from imblearn.over_sampling import SMOTE
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_val_score
from sklearn.metrics import (classification_report, confusion_matrix,
                             roc_auc_score, f1_score, recall_score)

print("Part 2: Model Training")

# Verify Part 1 output is available
try:
    print(f"\nUsing data from Part 1:")
    print(f"  X_train shape : {X_train_processed.shape}")
    print(f"  X_test shape  : {X_test_processed.shape}")
    print(f"  Train labels  : {dict(y_train.value_counts())} (0=fail, 1=pass)")
    print(f"  Test labels   : {dict(y_test.value_counts())} (0=fail, 1=pass)")
except NameError:
    raise RuntimeError("Part 1 variables not found. Please run part 1 first.")

# Apply SMOTE to balance training data
# SMOTE synthetically generates new minority-class examples (fails) by
# interpolating between existing fail cases: like a teacher creating
# practice exam questions based on the types of errors students already made.
#
# We only apply SMOTE to TRAINING data. Test data stays untouched so our
# evaluation reflects real-world class distribution.

print("\n Step 4: Applying SMOTE ")
print(f"  Before SMOTE → Pass: {(y_train==1).sum()}, Fail: {(y_train==0).sum()}")

smote = SMOTE(random_state=42, k_neighbors=5)
X_train_balanced, y_train_balanced = smote.fit_resample(X_train_processed, y_train)

print(f"  After SMOTE  → Pass: {(y_train_balanced==1).sum()}, Fail: {(y_train_balanced==0).sum()}")
print(f"  New training shape: {X_train_balanced.shape}")

# Baseline Model: Logistic Regression
# Logistic Regression is the "ruler" baseline. Before building a complex model,
# we need to know what a simple model can already achieve. If our Random Forest
# is only 1% better, the complexity isn't worth it.

print("\n Step 5: Training Baseline (Logistic Regression)")

lr_model = LogisticRegression(
    max_iter=1000,
    random_state=42,
    class_weight='balanced'   # extra safety on top of SMOTE
)
lr_model.fit(X_train_balanced, y_train_balanced)

lr_preds = lr_model.predict(X_test_processed)
lr_recall = recall_score(y_test, lr_preds, pos_label=0)
lr_f1     = f1_score(y_test, lr_preds, pos_label=0)
lr_auc    = roc_auc_score(y_test, lr_model.predict_proba(X_test_processed)[:, 0])

print(f"  Logistic Regression Results (failing class = 0):")
print(f"    Recall    : {lr_recall:.3f}  ← most important metric (catching at-risk trainees)")
print(f"    F1 Score  : {lr_f1:.3f}")
print(f"    AUC-ROC   : {lr_auc:.3f}")

# Main Model: Random Forest with GridSearchCV
# Random Forest is like asking 100 different doctors to independently assess
# a patient, then taking a majority vote. It's robust, handles mixed feature
# types well, and gives us feature importances to explain predictions.
#
# GridSearchCV tries every combination of hyperparameters systematically —
# like a chef trying every combination of ingredient quantities to find the
# best recipe.

print("\n Step 6: Training Main Model (Random Forest + GridSearchCV) ")
print("  This may take 1-2 minutes...")

# Define the hyperparameter grid to search
param_grid = {
    'n_estimators':      [100, 200, 300],    # how many trees in the forest
    'max_depth':         [5, 10, 15, None],  # how deep each tree grows
    'min_samples_split': [2, 5, 10],         # min samples needed to split a node
    'class_weight':      ['balanced']        # automatically handle class imbalance
}

rf_base = RandomForestClassifier(random_state=42)

# StratifiedKFold ensures each fold has proportional class representation
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

grid_search = GridSearchCV(
    estimator  = rf_base,
    param_grid = param_grid,
    cv         = cv,
    scoring    = 'recall',    # optimise for catching failing trainees
    n_jobs     = -1,          # use all CPU cores
    verbose    = 1
)

grid_search.fit(X_train_balanced, y_train_balanced)

best_rf = grid_search.best_estimator_
print(f"\n  Best Parameters found: {grid_search.best_params_}")

# Evaluate best RF on test set
rf_preds  = best_rf.predict(X_test_processed)
rf_proba  = best_rf.predict_proba(X_test_processed)[:, 0]  # probability of failing
rf_recall = recall_score(y_test, rf_preds, pos_label=0)
rf_f1     = f1_score(y_test, rf_preds, pos_label=0)
rf_auc    = roc_auc_score(y_test, rf_proba)

print(f"\n  Random Forest Results (failing class = 0):")
print(f"    Recall    : {rf_recall:.3f}")
print(f"    F1 Score  : {rf_f1:.3f}")
print(f"    AUC-ROC   : {rf_auc:.3f}")

# Full classification report
print(f"\n  Full Classification Report:")
print(classification_report(y_test, rf_preds, target_names=["At-Risk (Fail)", "On-Track (Pass)"]))

# Cross-Validation on balanced training data
# Checks if the model's performance is consistent or just
# got lucky with a particular train/test split, like retaking a test 5 times
# to confirm you genuinely understand the material.

print(" Step 7: Cross-Validation (5-Fold) ")
cv_scores = cross_val_score(
    best_rf, X_train_balanced, y_train_balanced,
    cv=cv, scoring='recall', n_jobs=-1
)
print(f"  CV Recall Scores: {[round(s,3) for s in cv_scores]}")
print(f"  Mean CV Recall  : {cv_scores.mean():.3f} (+/- {cv_scores.std():.3f})")

# Feature Importances
# Which features does the model rely on most? This tells us what actually
# predicts at-risk trainees, critical for the report's findings section.

print("\n Step 8: Feature Importances ")
feature_names = (
    numeric_cols +
    list(preprocessor.named_transformers_['cat']
         .get_feature_names_out(categorical_cols))
)
importances = pd.Series(
    best_rf.feature_importances_,
    index=feature_names
).sort_values(ascending=False)

print("  Top 10 Most Important Features:")
for feat, imp in importances.head(10).items():
    bar = "█" * int(imp * 100)
    print(f"    {feat:<35} {imp:.4f}  {bar}")

# Compare Baseline vs Main Model
print("\n── Step 9: Model Comparison Summary ──")
print(f"  {'Metric':<15} {'Logistic Regression':>22} {'Random Forest':>18}")
print(f"  {'─'*55}")
print(f"  {'Recall (Fail)':<15} {lr_recall:>22.3f} {rf_recall:>18.3f}")
print(f"  {'F1 (Fail)':<15} {lr_f1:>22.3f} {rf_f1:>18.3f}")
print(f"  {'AUC-ROC':<15} {lr_auc:>22.3f} {rf_auc:>18.3f}")

winner = "Random Forest" if rf_recall >= lr_recall else "Logistic Regression"
print(f"\n  Selected Model: {winner} (higher recall on failing class)")

# Save model artifacts
best_model = best_rf if rf_recall >= lr_recall else lr_model

joblib.dump(best_model,   'model.pkl')
joblib.dump(best_rf,      'random_forest_model.pkl')
joblib.dump(lr_model,     'logistic_regression_model.pkl')
joblib.dump(feature_names, 'feature_names.pkl')

# Save the processed test data for Part 3
joblib.dump(X_test_processed, 'X_test_processed.pkl')
joblib.dump(y_test,           'y_test.pkl')
joblib.dump(rf_proba,         'rf_proba.pkl')
joblib.dump(lr_model.predict_proba(X_test_processed)[:, 0], 'lr_proba.pkl')

print("\n Step 10: Saved Model Files ")
print("  model.pkl                  ← best overall model")
print("  random_forest_model.pkl    ← random forest (main)")
print("  logistic_regression_model.pkl ← baseline")
print("  feature_names.pkl          ← feature list for API")
print("  X_test_processed.pkl       ← test features for Part 3")
print("  y_test.pkl                 ← test labels for Part 3")
print("  rf_proba.pkl               ← RF probabilities for ROC curve")
print("  lr_proba.pkl               ← LR probabilities for ROC curve")

print("Part 2 COMPLETE")
print("Run part 3 next for evaluation plots and API.")

Part 2: Model Training

Using data from Part 1:
  X_train shape : (1156, 19)
  X_test shape  : (289, 19)
  Train labels  : {1: np.int64(1102), 0: np.int64(54)} (0=fail, 1=pass)
  Test labels   : {1: np.int64(281), 0: np.int64(8)} (0=fail, 1=pass)

 Step 4: Applying SMOTE 
  Before SMOTE → Pass: 1102, Fail: 54
  After SMOTE  → Pass: 1102, Fail: 1102
  New training shape: (2204, 19)

 Step 5: Training Baseline (Logistic Regression)
  Logistic Regression Results (failing class = 0):
    Recall    : 1.000  ← most important metric (catching at-risk trainees)
    F1 Score  : 0.485
    AUC-ROC   : 0.011

 Step 6: Training Main Model (Random Forest + GridSearchCV) 
  This may take 1-2 minutes...
Fitting 5 folds for each of 36 candidates, totalling 180 fits

  Best Parameters found: {'class_weight': 'balanced', 'max_depth': None, 'min_samples_split': 5, 'n_estimators': 300}

  Random Forest Results (failing class = 0):
    Recall    : 0.875
    F1 Score  : 0.560
    AUC-ROC   : 0.015

  Full Cl

In [3]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import joblib
import warnings
warnings.filterwarnings("ignore")

from sklearn.metrics import (
    confusion_matrix, classification_report,
    roc_curve, auc, precision_recall_curve,
    accuracy_score, precision_score, recall_score, f1_score
)

# Colors
BRAND_BLUE  = "#1F4E79"
BRAND_MID   = "#2E75B6"
BRAND_LIGHT = "#BDD7EE"
FAIL_RED    = "#C00000"
PASS_GREEN  = "#538135"
WARN_ORANGE = "#E97C30"
BG          = "#F7F9FC"

plt.rcParams.update({
    "figure.facecolor": BG, "axes.facecolor": BG,
    "axes.edgecolor":   "#CCCCCC", "axes.labelsize": 11,
    "axes.titlesize":   13, "axes.titleweight": "bold",
    "axes.titlecolor":  BRAND_BLUE, "font.family": "DejaVu Sans",
    "grid.color":       "#E0E0E0", "grid.linewidth": 0.7,
})

print("Part 3: Evaluation & API Integration")

# Load artifacts from Part 2
print("\n Step 1: Loading model artifacts ")
rf_model      = joblib.load('random_forest_model.pkl')
lr_model      = joblib.load('logistic_regression_model.pkl')
preprocessor  = joblib.load('hprime_preprocessor.pkl')
X_test_proc   = joblib.load('X_test_processed.pkl')
y_test        = joblib.load('y_test.pkl')
feature_names = joblib.load('feature_names.pkl')
print("  All artifacts loaded successfully.")

# Generate predictions
rf_preds     = rf_model.predict(X_test_proc)
lr_preds     = lr_model.predict(X_test_proc)
rf_fail_prob = rf_model.predict_proba(X_test_proc)[:, 0]
lr_fail_prob = lr_model.predict_proba(X_test_proc)[:, 0]

# Print full metrics
print("\n Step 3: Full Metrics Report ")
print("\n  RANDOM FOREST (Main Model):")
print(classification_report(y_test, rf_preds,
      target_names=["At-Risk (Fail)", "On-Track (Pass)"]))

print("  LOGISTIC REGRESSION (Baseline):")
print(classification_report(y_test, lr_preds,
      target_names=["At-Risk (Fail)", "On-Track (Pass)"]))

fpr_rf, tpr_rf, _ = roc_curve(y_test, rf_fail_prob, pos_label=0)
fpr_lr, tpr_lr, _ = roc_curve(y_test, lr_fail_prob, pos_label=0)
auc_rf = auc(fpr_rf, tpr_rf)
auc_lr = auc(fpr_lr, tpr_lr)

print(f"  AUC-ROC - Random Forest      : {auc_rf:.3f}")
print(f"  AUC-ROC - Logistic Regression: {auc_lr:.3f}")

# Evaluation Plots
print("\n Step 4: Generating Evaluation Plots ")

fig = plt.figure(figsize=(18, 12), facecolor=BG)
fig.suptitle("hPrime At-Risk Trainee Detection — Model Evaluation",
             fontsize=16, fontweight="bold", color=BRAND_BLUE, y=0.98)
gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.38)

# Confusion Matrix: Logistic Regression
ax = fig.add_subplot(gs[0, 0])
cm_lr = confusion_matrix(y_test, lr_preds)
sns.heatmap(cm_lr, annot=True, fmt="d", cmap="Blues", ax=ax,
            xticklabels=["At-Risk", "On-Track"],
            yticklabels=["At-Risk", "On-Track"],
            linewidths=1, linecolor="white",
            annot_kws={"size": 14, "weight": "bold"})
ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")
ax.set_title("Confusion Matrix\nLogistic Regression (Baseline)")

# Confusion Matrix: Random Forest
ax = fig.add_subplot(gs[0, 1])
cm_rf = confusion_matrix(y_test, rf_preds)
sns.heatmap(cm_rf, annot=True, fmt="d", cmap="OrRd", ax=ax,
            xticklabels=["At-Risk", "On-Track"],
            yticklabels=["At-Risk", "On-Track"],
            linewidths=1, linecolor="white",
            annot_kws={"size": 14, "weight": "bold"})
ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")
ax.set_title("Confusion Matrix\nRandom Forest (Main Model)")

# ROC Curves
ax = fig.add_subplot(gs[0, 2])
ax.plot(fpr_lr, tpr_lr, color=BRAND_MID, lw=2,
        label=f"Logistic Regression (AUC = {auc_lr:.3f})")
ax.plot(fpr_rf, tpr_rf, color=FAIL_RED, lw=2,
        label=f"Random Forest (AUC = {auc_rf:.3f})")
ax.plot([0,1],[0,1], "k--", lw=1.2, label="Random Classifier (AUC = 0.500)")
ax.fill_between(fpr_rf, tpr_rf, alpha=0.08, color=FAIL_RED)
ax.set_xlabel("False Positive Rate\n(Healthy trainees incorrectly flagged)")
ax.set_ylabel("True Positive Rate\n(At-risk trainees correctly caught)")
ax.set_title("ROC Curve Comparison")
ax.legend(fontsize=8); ax.grid()

# Precision-Recall Curve
ax = fig.add_subplot(gs[1, 0])
for fail_prob, label, color in [
    (lr_fail_prob, "Logistic Regression", BRAND_MID),
    (rf_fail_prob, "Random Forest",       FAIL_RED),
]:
    prec, rec, _ = precision_recall_curve(y_test, fail_prob, pos_label=0)
    pr_auc = auc(rec, prec)
    ax.plot(rec, prec, color=color, lw=2,
            label=f"{label} (AUC = {pr_auc:.3f})")
baseline_rate = (y_test == 0).mean()
ax.axhline(baseline_rate, color="#999", ls="--", lw=1.2,
           label=f"No-skill baseline ({baseline_rate:.1%} fail rate)")
ax.set_xlabel("Recall (At-risk trainees caught)")
ax.set_ylabel("Precision (Of flagged, truly at-risk)")
ax.set_title("Precision-Recall Curve")
ax.legend(fontsize=8); ax.grid()

# Feature Importances
ax = fig.add_subplot(gs[1, 1:])
importances = pd.Series(
    rf_model.feature_importances_,
    index=feature_names
).sort_values(ascending=True).tail(12)

n = len(importances)
bar_colors = []
for i in range(n):
    if i >= n - 3:   bar_colors.append(FAIL_RED)
    elif i >= n - 6: bar_colors.append(WARN_ORANGE)
    else:            bar_colors.append(BRAND_MID)

bars = ax.barh(importances.index, importances.values,
               color=bar_colors, edgecolor="white")
for bar, v in zip(bars, importances.values):
    ax.text(v + 0.001, bar.get_y() + bar.get_height()/2,
            f"{v:.4f}", va="center", fontsize=8)
ax.set_xlabel("Feature Importance Score")
ax.set_title("Top 12 Feature Importances — Random Forest")
ax.grid(axis="x")

from matplotlib.patches import Patch
ax.legend(handles=[
    Patch(color=FAIL_RED,    label="Top 3 predictors"),
    Patch(color=WARN_ORANGE, label="Strong predictors"),
    Patch(color=BRAND_MID,   label="Supporting features"),
], loc="lower right", fontsize=8)

plt.savefig("/content/evaluation_plots.png", dpi=150,
            bbox_inches="tight", facecolor=BG)
plt.show()
print("  Saved: evaluation_plots.png")

# Risk Score Demo Output
print("\n Step 5: Sample Risk Score Output ")
print("  (What supervisors would see in the hPrime dashboard)\n")

risk_scores = ((1 - rf_model.predict_proba(X_test_proc)[:, 1]) * 100).round(1)
risk_labels = pd.cut(risk_scores,
                     bins=[0, 25, 50, 75, 100],
                     labels=["Low Risk", "Moderate Risk",
                             "High Risk", "Critical Risk"])
output_df = pd.DataFrame({
    "Risk Score (0-100)": risk_scores[:15],
    "Risk Level":         risk_labels[:15],
    "Actual Outcome":     y_test.values[:15],
    "Predicted":          rf_preds[:15],
}).reset_index(drop=True)
output_df["Actual Outcome"] = output_df["Actual Outcome"].map(
    {0: "Failed", 1: "Passed"})
output_df["Predicted"] = output_df["Predicted"].map(
    {0: "At-Risk", 1: "On-Track"})
print(output_df.to_string(index=True))

# Save evaluate_model.py
print("\n Step 6: Saving evaluate_model.py ")
evaluate_script = '''"""
evaluate_model.py - Standalone Model Evaluation Script
CSCI323 Modern Artificial Intelligence, Spring 2026
University of Wollongong in Dubai

Run: python evaluate_model.py
Requires: model.pkl, hprime_preprocessor.pkl, hprime_clean.csv
"""
import pandas as pd
import numpy as np
import joblib
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, f1_score, recall_score,
    precision_score, accuracy_score, roc_curve, auc
)

print("Loading model and preprocessor...")
model        = joblib.load("model.pkl")
preprocessor = joblib.load("hprime_preprocessor.pkl")

df = pd.read_csv("hprime_clean.csv")
columns_to_drop = [
    "candidate_id","assessment_date","assessment_number","assessor_name",
    "assessor_title","competency_areas","focus_area","case_summary",
    "feedback","meets_standard","score_record_keeping","score_history_examination",
    "score_management_plan","score_clinical_judgement","score_history",
    "score_physical_exam","score_communication","score_clinical_judgement.1",
    "score_professionalism","score_organisation","score_overall_care",
    "avg_score","score_std","any_score_1"
]
y = df["meets_standard"]
X = df.drop(columns=columns_to_drop)
numeric_cols     = X.select_dtypes(include=["float64","int64"]).columns.tolist()
categorical_cols = X.select_dtypes(include=["object"]).columns.tolist()
X["days_since_last"] = X["days_since_last"].fillna(0)
X[numeric_cols] = X[numeric_cols].fillna(X[numeric_cols].median())
for col in categorical_cols:
    X[col] = X[col].fillna(X[col].mode()[0])

_, X_test, _, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_test_proc = preprocessor.transform(X_test)
preds       = model.predict(X_test_proc)
fail_proba  = model.predict_proba(X_test_proc)[:, 0]
fpr, tpr, _ = roc_curve(y_test, fail_proba, pos_label=0)

print("\\n" + "="*55)
print("MODEL EVALUATION REPORT")
print("At-Risk Junior Doctor Detection — hPrime Dataset")
print("="*55)
print(classification_report(y_test, preds,
      target_names=["At-Risk (Fail)","On-Track (Pass)"]))
print(f"AUC-ROC : {auc(fpr, tpr):.3f}")
print(f"Recall  : {recall_score(y_test, preds, pos_label=0):.3f}")
print(f"F1-Score: {f1_score(y_test, preds, pos_label=0, zero_division=0):.3f}")
'''
with open("/content/evaluate_model.py", "w") as f:
    f.write(evaluate_script)
print("  Saved: evaluate_model.py")

# Save app.py
print("\n Step 7: Saving app.py ")
api_code = '''"""
app.py - At-Risk Trainee Detection API
CSCI323 Modern Artificial Intelligence - Spring 2026
University of Wollongong in Dubai

Run: python app.py
Endpoint: POST http://localhost:5000/predict
"""
from flask import Flask, request, jsonify
import joblib
import pandas as pd

app = Flask(__name__)
model        = joblib.load("model.pkl")
preprocessor = joblib.load("hprime_preprocessor.pkl")

NUMERIC_COLS = [
    "patient_age_years","patient_is_female","is_emergency","days_since_last",
    "rolling_avg_3","rolling_fail_rate_3","cumulative_fail_count",
    "discussion_time_mins","prep_time_mins","feedback_length","num_competencies"
]
CATEGORICAL_COLS = ["form_type","assessor_seniority","setting"]


@app.route("/health", methods=["GET"])
def health():
    return jsonify({"status": "ok", "model": "At-Risk Trainee Detector v1.0"})


@app.route("/predict", methods=["POST"])
def predict():
    try:
        data = request.get_json()
        row  = {col: data.get(col, 0) for col in NUMERIC_COLS}
        row.update({col: data.get(col, "other") for col in CATEGORICAL_COLS})
        df          = pd.DataFrame([row])
        X_processed = preprocessor.transform(df)
        risk_prob   = float(model.predict_proba(X_processed)[0][0])
        risk_score  = round(risk_prob * 100, 1)
        if risk_score >= 75:   tier = "critical"
        elif risk_score >= 50: tier = "high"
        elif risk_score >= 25: tier = "moderate"
        else:                  tier = "low"
        recommendations = {
            "critical": "Immediate supervisor review required.",
            "high":     "Schedule check-in within the next assessment cycle.",
            "moderate": "Monitor closely in upcoming assessments.",
            "low":      "Trainee is progressing well. Continue standard monitoring."
        }
        return jsonify({
            "risk_score":     risk_score,
            "risk_tier":      tier,
            "prediction":     "at_risk" if risk_prob > 0.5 else "on_track",
            "recommendation": recommendations[tier]
        })
    except Exception as e:
        return jsonify({"error": str(e)}), 400


@app.route("/batch_predict", methods=["POST"])
def batch_predict():
    try:
        records = request.get_json()
        results = []
        for rec in records:
            row = {col: rec.get(col, 0) for col in NUMERIC_COLS}
            row.update({col: rec.get(col, "other") for col in CATEGORICAL_COLS})
            df          = pd.DataFrame([row])
            X_processed = preprocessor.transform(df)
            risk_prob   = float(model.predict_proba(X_processed)[0][0])
            risk_score  = round(risk_prob * 100, 1)
            results.append({
                "candidate_id": rec.get("candidate_id","unknown"),
                "risk_score":   risk_score,
                "risk_tier":    "critical" if risk_score>=75 else
                                "high"     if risk_score>=50 else
                                "moderate" if risk_score>=25 else "low",
                "prediction":   "at_risk" if risk_prob>0.5 else "on_track"
            })
        results.sort(key=lambda x: x["risk_score"], reverse=True)
        return jsonify({"total": len(results), "predictions": results})
    except Exception as e:
        return jsonify({"error": str(e)}), 400


if __name__ == "__main__":
    print("Starting hPrime At-Risk Trainee Detection API...")
    app.run(debug=True, port=5000)
'''
with open("/content/app.py", "w") as f:
    f.write(api_code)
print("  Saved: app.py")

# Done
print("Part 3 COMPLETE")
print("\nFiles saved to /content/ — check the Files panel on the left:")
print("  evaluation_plots.png")
print("  evaluate_model.py")
print("  app.py")
print("\nRight-click each file in the Files panel and click Download.")

Part 3: Evaluation & API Integration

 Step 1: Loading model artifacts 
  All artifacts loaded successfully.

 Step 3: Full Metrics Report 

  RANDOM FOREST (Main Model):
                 precision    recall  f1-score   support

 At-Risk (Fail)       0.41      0.88      0.56         8
On-Track (Pass)       1.00      0.96      0.98       281

       accuracy                           0.96       289
      macro avg       0.70      0.92      0.77       289
   weighted avg       0.98      0.96      0.97       289

  LOGISTIC REGRESSION (Baseline):
                 precision    recall  f1-score   support

 At-Risk (Fail)       0.32      1.00      0.48         8
On-Track (Pass)       1.00      0.94      0.97       281

       accuracy                           0.94       289
      macro avg       0.66      0.97      0.73       289
   weighted avg       0.98      0.94      0.96       289

  AUC-ROC - Random Forest      : 0.985
  AUC-ROC - Logistic Regression: 0.989

 Step 4: Generating Evalua